In [174]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [175]:
class PdData(Dataset):
    def __init__(self, DataFrame, feature_cols, label_col):
        self.data = DataFrame
        self.feature_cols = feature_cols
        self.label_col = label_col
        gender_map = {
            'male':0,
            'female':1
        }
        self.data['Sex'] = self.data['Sex'].map(gender_map)

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        feature = torch.tensor(self.data.iloc[idx, 2:],dtype = torch.float32)
        label = torch.tensor(self.data.iloc[idx, 1], dtype = torch.float32)
        return feature, label

    def all_labels(self):
        return torch.tensor(self.data.iloc[:,1].values, dtype = torch.float32)
    
    def all_feature(self):
        return torch.tensor(
            self.data.iloc[:,2:].values,
            dtype = torch.float32
        )
    
    def showdata(self):
        print(self.data)


In [176]:
DataFrame = pd.read_csv('./dataset/titanic/train.csv')
DataFrame = DataFrame.drop(['Cabin','Embarked','Name','Ticket'],axis = 1)
DataFrame.dropna(axis = 0, how = 'any', inplace=True)
DataFrame.columns


Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare'],
      dtype='object')

In [177]:
csv_file = './dataset/titanic/train.csv'
label_col = 1
feature_cols = [2,3,4,5,6,7,8]

dataset = PdData(DataFrame, feature_cols=feature_cols, label_col= label_col)
features = dataset.all_feature()
labels = dataset.all_labels()

In [181]:
batch_size = 100
lr = 0.0001
epoch = 500

net = torch.nn.Sequential(torch.nn.Linear(6,4),torch.nn.Linear(4,2),torch.nn.Linear(2,1))
loss = torch.nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(net.parameters(),lr)
iter = DataLoader(dataset, batch_size, shuffle=True)


for i in range(epoch):
    for x, y in iter:
        optim.zero_grad()
        y_hat = net(x)
        l = loss(y_hat, y.reshape(y_hat.shape))
        l.backward()
        optim.step()

    with torch.no_grad():
        out = net(features)
        l = loss(out, labels.reshape(out.shape))
        print(f'epoch:{i + 1}, loss:{float(l)}')


epoch:1, loss:2.406583070755005
epoch:2, loss:2.3196849822998047
epoch:3, loss:2.2415213584899902
epoch:4, loss:2.170081615447998
epoch:5, loss:2.1052119731903076
epoch:6, loss:2.0459766387939453
epoch:7, loss:1.988339900970459
epoch:8, loss:1.9251418113708496
epoch:9, loss:1.8726445436477661
epoch:10, loss:1.8243681192398071
epoch:11, loss:1.7721658945083618
epoch:12, loss:1.7322065830230713
epoch:13, loss:1.6941039562225342
epoch:14, loss:1.6483980417251587
epoch:15, loss:1.6132175922393799
epoch:16, loss:1.5784821510314941
epoch:17, loss:1.5447766780853271
epoch:18, loss:1.5159416198730469
epoch:19, loss:1.4888581037521362
epoch:20, loss:1.461652159690857
epoch:21, loss:1.4371248483657837
epoch:22, loss:1.4153988361358643
epoch:23, loss:1.3908436298370361
epoch:24, loss:1.3679420948028564
epoch:25, loss:1.349328875541687
epoch:26, loss:1.3282809257507324
epoch:27, loss:1.3105050325393677
epoch:28, loss:1.2926855087280273
epoch:29, loss:1.2772268056869507
epoch:30, loss:1.25823748111

In [182]:
torch.save(net, "./titanic.pth")